# Tâche 1 — v14 corrigé

Version corrigée de la meilleure idée de `pres_v14.ipynb` :

- **split par document** avant toute création de chunks
- **développement local** avec `train / val / test_local`
- **sauvegarde du meilleur modèle** dans un répertoire paramétrable
- **prédiction locale** sur `test_local`
- **prédiction finale** sur le vrai fichier test si vous l’ajoutez

Le but est d'éviter une validation trop optimiste due à un split par chunks.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Installation

In [2]:
!pip install -q transformers datasets accelerate scikit-learn torch

## 2. Configuration
Remplacez simplement les chemins par les vôtres si besoin.

In [3]:

import os
from pathlib import Path

# === FICHIERS ===
TRAIN_FILE = "/content/drive/MyDrive/projet tal/corpus.tache1.learn.utf8"      # ou votre nom local
#TEST_FILE  = "corpus_tache1_test.utf8"       # optionnel pour la soumission finale

# === DOSSIERS DE SORTIE ===
# Remplacez par un chemin Drive/Colab si vous voulez persister le modèle.
MODEL_SAVE_DIR = "/content/drive/MyDrive/projet tal/rital_tache-doc/models/v14_corrige_best"
OUTPUT_DIR     = "/content/drive/MyDrive/projet tal/rital_tache-doc/runs/v14_corrige"
SUBMISSION_DIR = "/content/drive/MyDrive/projet tal/rital_tache-doc/submissions"

Path(MODEL_SAVE_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(SUBMISSION_DIR).mkdir(parents=True, exist_ok=True)

print("MODEL_SAVE_DIR =", MODEL_SAVE_DIR)
print("OUTPUT_DIR     =", OUTPUT_DIR)
print("SUBMISSION_DIR =", SUBMISSION_DIR)


MODEL_SAVE_DIR = /content/drive/MyDrive/projet tal/rital_tache-doc/models/v14_corrige_best
OUTPUT_DIR     = /content/drive/MyDrive/projet tal/rital_tache-doc/runs/v14_corrige
SUBMISSION_DIR = /content/drive/MyDrive/projet tal/rital_tache-doc/submissions


## 3. Imports

In [4]:

import os, re, codecs, gc, json, random
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset
from sklearn.metrics import recall_score, precision_score

from sklearn.metrics import (
    f1_score, average_precision_score, roc_auc_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from datasets import Dataset as HFDataset

os.environ["WANDB_DISABLED"] = "true"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


## 4. Fonctions utilitaires

In [5]:

def read_train_corpus(path):
    entries = []
    with codecs.open(path, "r", "utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = re.match(r"<(\d+):(\d+):([CM])>\s*(.*)", line)
            if m:
                entries.append({
                    "doc": int(m.group(1)),
                    "sent": int(m.group(2)),
                    "label_char": m.group(3),
                    "label": 1 if m.group(3) == "M" else 0,
                    "text": m.group(4),
                })
    return entries

def read_test_corpus(path):
    entries = []
    with codecs.open(path, "r", "utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = re.match(r"<(\d+):(\d+)>\s*(.*)", line)
            if m:
                entries.append({
                    "doc": int(m.group(1)),
                    "sent": int(m.group(2)),
                    "text": m.group(3),
                })
    return entries

def summarize_entries(entries, name="set"):
    n_docs = len({e["doc"] for e in entries})
    cnt = Counter(e["label_char"] for e in entries if "label_char" in e)
    print(f"[{name}] docs={n_docs:,} phrases={len(entries):,}")
    if cnt:
        print(f"  C={cnt.get('C',0):,} | M={cnt.get('M',0):,} | M%={100*cnt.get('M',0)/max(1,sum(cnt.values())):.2f}")

def build_doc_label_profile(entries):
    by_doc = defaultdict(set)
    for e in entries:
        by_doc[e["doc"]].add(e["label_char"])
    pure_c = sum(v == {"C"} for v in by_doc.values())
    pure_m = sum(v == {"M"} for v in by_doc.values())
    mixed  = sum(v == {"C","M"} for v in by_doc.values())
    print(f"Documents: purs C={pure_c}, purs M={pure_m}, mixtes={mixed}")

def build_chunks_from_entries(entries, max_words=350):
    chunks = []
    current_doc = None
    current_label = None
    current_texts = []
    current_words = 0

    for entry in entries:
        words = len(entry["text"].split())
        same_segment = (
            entry["doc"] == current_doc and
            entry["label"] == current_label and
            current_words + words <= max_words
        )
        if same_segment:
            current_texts.append(entry["text"])
            current_words += words
        else:
            if current_texts:
                chunks.append({
                    "text": " ".join(current_texts),
                    "label": current_label,
                    "n_sents": len(current_texts),
                    "doc": current_doc,
                })
            current_doc = entry["doc"]
            current_label = entry["label"]
            current_texts = [entry["text"]]
            current_words = words

    if current_texts:
        chunks.append({
            "text": " ".join(current_texts),
            "label": current_label,
            "n_sents": len(current_texts),
            "doc": current_doc,
        })
    return chunks

def build_context_windows(entries, max_context_words=350):
    by_doc = defaultdict(list)
    for i, entry in enumerate(entries):
        by_doc[entry["doc"]].append((i, entry["text"]))

    windows = []
    for doc_id in sorted(by_doc.keys()):
        sents = sorted(by_doc[doc_id], key=lambda x: x[0])
        for pos, (global_idx, text) in enumerate(sents):
            window = [text]
            word_count = len(text.split())
            left, right = pos - 1, pos + 1

            while word_count < max_context_words:
                added = False
                if left >= 0:
                    w = len(sents[left][1].split())
                    if word_count + w <= max_context_words:
                        window.insert(0, sents[left][1])
                        word_count += w
                        left -= 1
                        added = True
                if right < len(sents):
                    w = len(sents[right][1].split())
                    if word_count + w <= max_context_words:
                        window.append(sents[right][1])
                        word_count += w
                        right += 1
                        added = True
                if not added:
                    break
            windows.append((global_idx, " ".join(window)))

    windows = sorted(windows, key=lambda x: x[0])
    return [w[1] for w in windows]

def label_distribution_from_docs(entries, doc_ids):
    sub = [e for e in entries if e["doc"] in set(doc_ids)]
    cnt = Counter(e["label_char"] for e in sub)
    return cnt, len(sub)

def print_split_stats(name, entries):
    summarize_entries(entries, name=name)
    build_doc_label_profile(entries)

class ChunkDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

    def __len__(self):
        return len(self.texts)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs > 0.5).astype(int)
    f1m = f1_score(labels, preds, average="macro")
    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0
    ap  = average_precision_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0
    return {
        "f1_macro": round(f1m * 100, 3),
        "auc": round(auc * 100, 3),
        "ap": round(ap * 100, 3),
    }

def evaluate_probs(y_true, probs, title="eval"):
    preds = (probs > 0.5).astype(int)
    f1m = f1_score(y_true, preds, average="macro")
    auc = roc_auc_score(y_true, probs) if len(np.unique(y_true)) > 1 else 0.0
    ap  = average_precision_score(y_true, probs) if len(np.unique(y_true)) > 1 else 0.0
    cm = confusion_matrix(y_true, preds)
    print(f"\n[{title}] F1_macro={f1m*100:.2f} | AUC={auc*100:.2f} | AP={ap*100:.2f}")
    print("Confusion matrix [[TN, FP],[FN, TP]]:")
    print(cm)
    print(classification_report(y_true, preds, target_names=["Chirac", "Mitterrand"], digits=4))
    return {"f1_macro": f1m, "auc": auc, "ap": ap, "cm": cm}


## 5. Chargement du corpus d'entraînement

In [6]:

train_entries = read_train_corpus(TRAIN_FILE)
summarize_entries(train_entries, "train_full")
build_doc_label_profile(train_entries)

train_df = pd.DataFrame(train_entries)
train_df.head()


[train_full] docs=587 phrases=57,413
  C=49,890 | M=7,523 | M%=13.10
Documents: purs C=187, purs M=0, mixtes=400


,doc,sent,label_char,label,text
0,100,1,C,0,"Quand je dis chers amis, il ne s'agit pas là d..."
1,100,2,C,0,D'abord merci de cet exceptionnel accueil que ...
2,100,3,C,0,C'est toujours très émouvant de venir en Afriq...
3,100,4,C,0,Aucun citoyen français ne peut être indifféren...
4,100,5,C,0,"Le Congo, que naguère le <nom> qualifia de ""re..."


## 6. Split par document : train / val / test_local

In [7]:

# On split les doc_id, pas les phrases ni les chunks.
doc_ids = sorted(train_df["doc"].unique())

doc_meta = (
    train_df.groupby("doc")
    .agg(
        n_phrases=("text", "size"),
        n_M=("label", "sum"),
    )
    .reset_index()
)
doc_meta["has_M"] = (doc_meta["n_M"] > 0).astype(int)

# 80 / 10 / 10 via deux splits stratifiés approximatifs sur has_M
docs_train, docs_temp = train_test_split(
    doc_meta["doc"].values,
    test_size=0.20,
    random_state=SEED,
    stratify=doc_meta["has_M"].values,
)

temp_meta = doc_meta.set_index("doc").loc[docs_temp].reset_index()
docs_val, docs_test_local = train_test_split(
    temp_meta["doc"].values,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_meta["has_M"].values,
)

docs_train = set(docs_train)
docs_val = set(docs_val)
docs_test_local = set(docs_test_local)

train_entries_split = [e for e in train_entries if e["doc"] in docs_train]
val_entries_split = [e for e in train_entries if e["doc"] in docs_val]
test_local_entries = [e for e in train_entries if e["doc"] in docs_test_local]

print_split_stats("train_split", train_entries_split)
print_split_stats("val_split", val_entries_split)
print_split_stats("test_local_split", test_local_entries)


[train_split] docs=469 phrases=46,079
  C=40,077 | M=6,002 | M%=13.03
Documents: purs C=149, purs M=0, mixtes=320
[val_split] docs=59 phrases=5,458
  C=4,725 | M=733 | M%=13.43
Documents: purs C=19, purs M=0, mixtes=40
[test_local_split] docs=59 phrases=5,876
  C=5,088 | M=788 | M%=13.41
Documents: purs C=19, purs M=0, mixtes=40


## 7. Création des chunks train/val et des fenêtres test_local

In [8]:

MAX_WORDS = 350
MAX_CONTEXT_WORDS = 350

train_chunks = build_chunks_from_entries(train_entries_split, max_words=MAX_WORDS)
val_chunks = build_chunks_from_entries(val_entries_split, max_words=MAX_WORDS)

print(f"Train chunks: {len(train_chunks):,}")
print(f"Val chunks:   {len(val_chunks):,}")
print(f"Avg sents/chunk train: {np.mean([c['n_sents'] for c in train_chunks]):.2f}")
print(f"Avg words/chunk train: {np.mean([len(c['text'].split()) for c in train_chunks]):.2f}")

train_texts = [c["text"] for c in train_chunks]
train_labels = np.array([c["label"] for c in train_chunks])

val_texts = [c["text"] for c in val_chunks]
val_labels = np.array([c["label"] for c in val_chunks])

test_local_window_texts = build_context_windows(test_local_entries, max_context_words=MAX_CONTEXT_WORDS)
test_local_labels = np.array([e["label"] for e in test_local_entries])

print("Local test windows:", len(test_local_window_texts))
print("Local test labels :", len(test_local_labels))
assert len(test_local_window_texts) == len(test_local_labels)


Train chunks: 3,445
Val chunks:   417
Avg sents/chunk train: 13.38
Avg words/chunk train: 287.49
Local test windows: 5876
Local test labels : 5876


## 8. Tokenizer + datasets

In [9]:

MODEL_NAME = "camembert/camembert-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = ChunkDataset(train_texts, train_labels, tokenizer, max_length=512)
val_dataset   = ChunkDataset(val_texts, val_labels, tokenizer, max_length=512)
print("Datasets prêts.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/809k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

Datasets prêts.


## 9. Entraînement

In [10]:

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="auc",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: camembert/camembert-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Macro,Auc,Ap
1,1.346651,0.090486,95.977000,99.387000,98.244000
2,0.611485,0.073504,97.561000,99.735000,99.117000
3,0.353595,0.044253,98.038000,99.890000,99.631000
4,0.130113,0.076702,97.586000,99.871000,99.504000
5,0.022590,0.153575,96.714000,99.756000,99.081000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=540, training_loss=0.4569383286215641, metrics={'train_runtime': 2004.828, 'train_samples_per_second': 17.184, 'train_steps_per_second': 0.539, 'total_flos': 1.60525177333248e+16, 'train_loss': 0.4569383286215641, 'epoch': 5.0})

## 10. Évaluation sur les chunks de validation

In [11]:

val_results = trainer.evaluate()
print("\nBEST CHECKPOINT (validation chunks):")
for k, v in val_results.items():
    if k.startswith("eval_"):
        print(f"  {k.replace('eval_', ''):>12s} : {v}")



BEST CHECKPOINT (validation chunks):
          loss : 0.04406611993908882
      f1_macro : 98.038
           auc : 99.89
            ap : 99.631
       runtime : 12.9159
  samples_per_second : 32.286
  steps_per_second : 2.09


## 11. Sauvegarde du meilleur modèle et du tokenizer

In [ ]:

trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print("Modèle sauvegardé dans:", MODEL_SAVE_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvegardé dans: /content/drive/MyDrive/projet tal/rital_tache-doc/models/v14_corrige_best


## 12. Évaluation locale réaliste sur `test_local`
On applique la stratégie v14 : prédiction de fenêtres de contexte sur un split jamais vu en entraînement.

In [12]:

test_local_hf = HFDataset.from_dict({"text": test_local_window_texts})

def tokenize_fn(ex):
    return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=512)

test_local_tok = test_local_hf.map(tokenize_fn, batched=True, batch_size=128)
pred_local = trainer.predict(test_local_tok)
logits_local = pred_local.predictions
p_local = torch.nn.functional.softmax(torch.from_numpy(logits_local), dim=-1)[:, 1].numpy()

local_metrics = evaluate_probs(test_local_labels, p_local, title="test_local_windows")

print("\nDistribution des probabilités:")
print(f"  < 0.1:   {np.sum(p_local < 0.1):>6} ({np.mean(p_local < 0.1)*100:.1f}%)")
print(f"  > 0.9:   {np.sum(p_local > 0.9):>6} ({np.mean(p_local > 0.9)*100:.1f}%)")
print(f"  0.1-0.9: {np.sum((p_local >= 0.1) & (p_local <= 0.9)):>6} ({np.mean((p_local >= 0.1) & (p_local <= 0.9))*100:.1f}%)")


Map:   0%|          | 0/5876 [00:00<?, ? examples/s]


[test_local_windows] F1_macro=87.91 | AUC=97.21 | AP=83.12
Confusion matrix [[TN, FP],[FN, TP]]:
[[4856  232]
 [ 118  670]]
              precision    recall  f1-score   support

      Chirac     0.9763    0.9544    0.9652      5088
  Mitterrand     0.7428    0.8503    0.7929       788

    accuracy                         0.9404      5876
   macro avg     0.8595    0.9023    0.8791      5876
weighted avg     0.9450    0.9404    0.9421      5876


Distribution des probabilités:
  < 0.1:     4959 (84.4%)
  > 0.9:      859 (14.6%)
  0.1-0.9:     58 (1.0%)


## 12bis. Analyse d'erreurs sur `test_local`

Les cellules suivantes servent à comprendre **où** la version chunks se trompe :

- erreurs par label ;
- documents mixtes vs purs ;
- influence de la longueur de phrase ;
- influence de la proximité d'une frontière de label ;
- calibration grossière par zones de probabilité.

Elles n'affectent pas l'entraînement : elles servent uniquement au diagnostic.


In [14]:

# Table détaillée des prédictions locales phrase par phrase
test_local_df = pd.DataFrame(test_local_entries).copy()
test_local_df["window_text"] = test_local_window_texts
test_local_df["p_mitterrand"] = p_local
test_local_df["pred"] = (test_local_df["p_mitterrand"] > 0.5).astype(int)
test_local_df["correct"] = (test_local_df["pred"] == test_local_df["label"]).astype(int)
test_local_df["error"] = 1 - test_local_df["correct"]
test_local_df["n_words_phrase"] = test_local_df["text"].str.split().str.len()
test_local_df["n_words_window"] = test_local_df["window_text"].str.split().str.len()

# Type de document : pur C, pur M ou mixte
doc_label_sets = (
    pd.DataFrame(train_entries)
    .groupby("doc")["label_char"]
    .agg(lambda s: "".join(sorted(set(s))))
    .to_dict()
)

def doc_type_from_set(v):
    if v == "C":
        return "pur_C"
    if v == "M":
        return "pur_M"
    return "mixte"

test_local_df["doc_type"] = test_local_df["doc"].map(lambda d: doc_type_from_set(doc_label_sets.get(d, "CM")))

# Position et frontière de label dans le document
test_local_df = test_local_df.sort_values(["doc", "sent"]).reset_index(drop=True)
test_local_df["prev_label"] = test_local_df.groupby("doc")["label"].shift(1)
test_local_df["next_label"] = test_local_df.groupby("doc")["label"].shift(-1)
test_local_df["is_boundary"] = (
    ((test_local_df["prev_label"].notna()) & (test_local_df["prev_label"] != test_local_df["label"])) |
    ((test_local_df["next_label"].notna()) & (test_local_df["next_label"] != test_local_df["label"]))
).astype(int)

# Distance à la frontière la plus proche (en nombre de phrases)
dist_to_boundary = []
for doc_id, g in test_local_df.groupby("doc", sort=False):
    idxs = list(g.index)
    boundary_idxs = [i for i in idxs if test_local_df.loc[i, "is_boundary"] == 1]
    if not boundary_idxs:
        dist_to_boundary.extend([999] * len(idxs))
    else:
        for i in idxs:
            dist_to_boundary.append(min(abs(i - b) for b in boundary_idxs))

test_local_df["dist_boundary"] = dist_to_boundary
test_local_df["near_boundary_le_1"] = (test_local_df["dist_boundary"] <= 1).astype(int)
test_local_df["near_boundary_le_3"] = (test_local_df["dist_boundary"] <= 3).astype(int)

print("Table d'analyse prête :", test_local_df.shape)
test_local_df.head()


Table d'analyse prête : (5876, 19)


,doc,sent,label_char,label,text,window_text,p_mitterrand,pred,correct,error,n_words_phrase,n_words_window,doc_type,prev_label,next_label,is_boundary,dist_boundary,near_boundary_le_1,near_boundary_le_3
0,15,1,C,0,<nom> était l'honneur des Nations Unies.,<nom> était l'honneur des Nations Unies. Tombé...,0.000792,0,1,0,6,342,mixte,NaN,0.0,0,18,0,0
1,15,2,C,0,"Tombé le 19 août avec ses collaborateurs, il r...",<nom> était l'honneur des Nations Unies. Tombé...,0.000792,0,1,0,12,342,mixte,0.0,0.0,0,17,0,0
2,15,3,C,0,Dédions cette session à ce grand serviteur du ...,<nom> était l'honneur des Nations Unies. Tombé...,0.000792,0,1,0,13,342,mixte,0.0,0.0,0,16,0,0
3,15,4,C,0,Les Nations Unies viennent de traverser l'une ...,<nom> était l'honneur des Nations Unies. Tombé...,0.000792,0,1,0,15,342,mixte,0.0,0.0,0,15,0,0
4,15,5,C,0,"Le respect de la Charte, l'usage de la force o...",<nom> était l'honneur des Nations Unies. Tombé...,0.000792,0,1,0,15,342,mixte,0.0,0.0,0,14,0,0


### 12bis.a Vue d'ensemble des erreurs

In [15]:

overall = pd.DataFrame({
    "n": [len(test_local_df)],
    "accuracy": [test_local_df["correct"].mean()],
    "error_rate": [test_local_df["error"].mean()],
    "mean_p_mitterrand": [test_local_df["p_mitterrand"].mean()],
    "mean_words_phrase": [test_local_df["n_words_phrase"].mean()],
    "mean_words_window": [test_local_df["n_words_window"].mean()],
})
overall


,n,accuracy,error_rate,mean_p_mitterrand,mean_words_phrase,mean_words_window
0,5876,0.940436,0.059564,0.152856,21.893805,340.595643


### 12bis.b Erreurs par label et par type de document

In [16]:

by_label = (
    test_local_df.groupby("label_char")
    .agg(
        n=("label", "size"),
        error_rate=("error", "mean"),
        recall_proxy=("correct", "mean"),
        mean_p_mitterrand=("p_mitterrand", "mean"),
        mean_words_phrase=("n_words_phrase", "mean"),
    )
    .reset_index()
)
print("Par label :")
display(by_label)

by_doc_type = (
    test_local_df.groupby("doc_type")
    .agg(
        n=("label", "size"),
        error_rate=("error", "mean"),
        mean_p_mitterrand=("p_mitterrand", "mean"),
        mean_words_phrase=("n_words_phrase", "mean"),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)
print("Par type de document :")
display(by_doc_type)

cross = (
    test_local_df.groupby(["doc_type", "label_char"])
    .agg(
        n=("label", "size"),
        error_rate=("error", "mean"),
        mean_p_mitterrand=("p_mitterrand", "mean"),
    )
    .reset_index()
)
print("Croisement type de document × label :")
display(cross)


Par label :


,label_char,n,error_rate,recall_proxy,mean_p_mitterrand,mean_words_phrase
0,C,5088,0.045597,0.954403,0.045602,21.343160
1,M,788,0.149746,0.850254,0.845382,25.449239


Par type de document :


,doc_type,n,error_rate,mean_p_mitterrand,mean_words_phrase
0,mixte,3609,0.096703,0.248023,22.980881
1,pur_C,2267,0.000441,0.001354,20.163211


Croisement type de document × label :


,doc_type,label_char,n,error_rate,mean_p_mitterrand
0,mixte,C,2821,0.081886,0.081160
1,mixte,M,788,0.149746,0.845382
2,pur_C,C,2267,0.000441,0.001354


### 12bis.c Erreurs près des frontières de label

In [17]:

boundary_stats = (
    test_local_df.groupby("near_boundary_le_1")
    .agg(
        n=("label", "size"),
        error_rate=("error", "mean"),
        mean_p_mitterrand=("p_mitterrand", "mean"),
        mean_words_phrase=("n_words_phrase", "mean"),
    )
    .reset_index()
)
boundary_stats["zone"] = boundary_stats["near_boundary_le_1"].map({0: "loin_frontiere", 1: "a_1_phrase_ou_moins"})
display(boundary_stats[["zone", "n", "error_rate", "mean_p_mitterrand", "mean_words_phrase"]])

boundary_stats3 = (
    test_local_df.groupby("near_boundary_le_3")
    .agg(
        n=("label", "size"),
        error_rate=("error", "mean"),
        mean_p_mitterrand=("p_mitterrand", "mean"),
    )
    .reset_index()
)
boundary_stats3["zone"] = boundary_stats3["near_boundary_le_3"].map({0: "loin_frontiere", 1: "a_3_phrases_ou_moins"})
display(boundary_stats3[["zone", "n", "error_rate", "mean_p_mitterrand"]])

print("Distance moyenne à la frontière :")
display(
    test_local_df.groupby("error")["dist_boundary"].agg(["count", "mean", "median"]).rename(index={0:"correct",1:"error"})
)


,zone,n,error_rate,mean_p_mitterrand,mean_words_phrase
0,loin_frontiere,5557,0.038510,0.124869,21.646572
1,a_1_phrase_ou_moins,319,0.426332,0.640391,26.200627


,zone,n,error_rate,mean_p_mitterrand
0,loin_frontiere,5250,0.021905,0.096161
1,a_3_phrases_ou_moins,626,0.375399,0.628337


Distance moyenne à la frontière :


,count,mean,median
error,,,
correct,5526,427.493847,60.0
error,350,6.848571,2.0


### 12bis.d Longueur des phrases et confiance du modèle

In [18]:

test_local_df["len_bin"] = pd.cut(
    test_local_df["n_words_phrase"],
    bins=[0, 5, 10, 20, 40, 1000],
    labels=["1-5", "6-10", "11-20", "21-40", "41+"],
    include_lowest=True,
)

len_stats = (
    test_local_df.groupby("len_bin", observed=False)
    .agg(
        n=("label", "size"),
        error_rate=("error", "mean"),
        mean_p_mitterrand=("p_mitterrand", "mean"),
    )
    .reset_index()
)
display(len_stats)

test_local_df["conf_zone"] = pd.cut(
    test_local_df["p_mitterrand"],
    bins=[-0.001, 0.1, 0.4, 0.6, 0.9, 1.001],
    labels=["<0.1", "0.1-0.4", "0.4-0.6", "0.6-0.9", ">0.9"],
    include_lowest=True,
)

conf_stats = (
    test_local_df.groupby("conf_zone", observed=False)
    .agg(
        n=("label", "size"),
        error_rate=("error", "mean"),
        prop_m_true=("label", "mean"),
    )
    .reset_index()
)
display(conf_stats)


,len_bin,n,error_rate,mean_p_mitterrand
0,1-5,281,0.042705,0.137442
1,6-10,936,0.063034,0.146256
2,11-20,2019,0.052006,0.131429
3,21-40,2098,0.069590,0.154595
4,41+,542,0.051661,0.245334


,conf_zone,n,error_rate,prop_m_true
0,<0.1,4959,0.022989,0.022989
1,0.1-0.4,14,0.214286,0.214286
2,0.4-0.6,11,0.454545,0.636364
3,0.6-0.9,33,0.787879,0.212121
4,>0.9,859,0.235157,0.764843


### 12bis.e Exemples d'erreurs à inspecter

In [19]:

# Erreurs Mitterrand ratées (FN) et Chirac confondu avec Mitterrand (FP)
false_negatives = test_local_df[(test_local_df["label"] == 1) & (test_local_df["pred"] == 0)].copy()
false_positives = test_local_df[(test_local_df["label"] == 0) & (test_local_df["pred"] == 1)].copy()

print("Exemples de faux négatifs (vrai Mitterrand, prédit Chirac) :")
display(false_negatives[["doc", "sent", "label_char", "p_mitterrand", "doc_type", "dist_boundary", "n_words_phrase", "text"]].head(15))

print("Exemples de faux positifs (vrai Chirac, prédit Mitterrand) :")
display(false_positives[["doc", "sent", "label_char", "p_mitterrand", "doc_type", "dist_boundary", "n_words_phrase", "text"]].head(15))


Exemples de faux négatifs (vrai Mitterrand, prédit Chirac) :


,doc,sent,label_char,p_mitterrand,doc_type,dist_boundary,n_words_phrase,text
133,88,11,M,0.001034,mixte,0,22,"Tout cela forme un tout, nous n'en avons pas f..."
204,98,57,M,0.060087,mixte,0,53,Mais il n'est pas mauvais qu'on le répète sans...
443,151,45,M,0.011778,mixte,0,19,"L'Afrique, on le sait, et c'était fort bien di..."
445,151,47,M,0.451795,mixte,2,17,"Dans certains de vos pays, le niveau de vie a ..."
824,160,9,M,0.000951,mixte,0,41,C'est précisément pourquoi le devoir d'attenti...
825,160,10,M,0.000951,mixte,1,2,Vaste programme.
826,160,11,M,0.003601,mixte,2,27,"L'éternelle revendication du bonheur, par défi..."
827,160,12,M,0.004512,mixte,3,15,Il doit simplement veiller à ce que les condit...
828,160,13,M,0.004512,mixte,4,20,Il y a une dynamique de 1789 qui déborde l'évé...
829,160,14,M,0.004512,mixte,5,39,Comme la Déclaration des droits 'de l'homme' s...


Exemples de faux positifs (vrai Chirac, prédit Mitterrand) :


,doc,sent,label_char,p_mitterrand,doc_type,dist_boundary,n_words_phrase,text
15,15,16,C,0.938294,mixte,3,13,"D'abord, le règlement des conflits qui menacen..."
16,15,17,C,0.996937,mixte,2,26,"En Iraq, le transfert de la souveraineté aux I..."
17,15,18,C,0.997311,mixte,1,11,Il appartient à l'ONU de donner sa légitimité ...
18,15,19,C,0.997311,mixte,0,37,C'est aussi à l'ONU qu'il revient d'accompagne...
147,88,25,C,0.998700,mixte,0,24,"Je rends hommage au <nom> et à M. <nom> , qui ..."
182,98,35,C,0.996954,mixte,0,13,"Il faut donc pouvoir entreprendre, de façon pl..."
312,130,60,C,0.997973,mixte,7,17,"Cet effort sera poursuivi en <date>, et les pr..."
313,130,61,C,0.998303,mixte,6,5,Je m'y suis engagé personnellement.
314,130,62,C,0.998283,mixte,5,9,Mais la France a aussi besoin de réformes stru...
315,130,63,C,0.998525,mixte,4,6,C'est la voie que j'ai choisie.


## 12ter. Inspection du contexte des erreurs

On affiche ici les erreurs avec leur **contexte réel dans le document** pour vérifier l'hypothèse suivante :

- la phrase centrale est correcte seule ou stylistiquement marquée,
- mais le **contexte dominant autour** pousse le modèle vers l'autre président,
- surtout dans les **documents mixtes** et près des **frontières de label**.

Ces cellules ne réentraînent pas le modèle : elles servent à **voir** les cas difficiles.


In [20]:

from collections import defaultdict

def build_entries_by_doc(entries):
    by_doc = defaultdict(list)
    for e in entries:
        by_doc[e["doc"]].append(e)
    for doc_id in by_doc:
        by_doc[doc_id] = sorted(by_doc[doc_id], key=lambda x: x["sent"])
    return by_doc

entries_by_doc_test_local = build_entries_by_doc(test_local_entries)

def show_error_context_examples(
    pred_df,
    entries_by_doc,
    n=12,
    window=3,
    near_boundary_only=False,
    mixed_only=False,
    random_state=42,
):
    df = pred_df[pred_df["error"] == 1].copy()

    if near_boundary_only and "near_boundary_le_1" in df.columns:
        df = df[df["near_boundary_le_1"] == 1]

    if mixed_only and "doc_type" in df.columns:
        df = df[df["doc_type"] == "mixte"]

    print(f"Erreurs retenues: {len(df)}")
    if len(df) == 0:
        print("Aucune erreur à afficher.")
        return

    sample = df.sample(n=min(n, len(df)), random_state=random_state)

    for _, row in sample.iterrows():
        doc_id = int(row["doc"])
        sent_id = int(row["sent"])
        doc_entries = entries_by_doc[doc_id]
        pos = None
        for i, e in enumerate(doc_entries):
            if int(e["sent"]) == sent_id:
                pos = i
                break

        print("=" * 120)
        print(
            f"DOC={doc_id} | SENT={sent_id} | doc_type={row.get('doc_type', 'NA')} | "
            f"dist_frontiere={row.get('dist_to_boundary', 'NA')} | "
            f"TRUE={row['label_char']} | PRED={'M' if row['pred']==1 else 'C'} | "
            f"p(M)={row['p_mitterrand']:.4f}"
        )
        print("-" * 120)

        left = max(0, pos - window)
        right = min(len(doc_entries), pos + window + 1)

        for j in range(left, right):
            e = doc_entries[j]
            marker = ">>> " if j == pos else "    "
            print(f"{marker}[sent={e['sent']:>3}] TRUE={e['label_char']} | {e['text']}")
        print()

print("Fonctions d'inspection du contexte prêtes.")


Fonctions d'inspection du contexte prêtes.


In [21]:

# Exemples d'erreurs proches des frontières dans des documents mixtes
show_error_context_examples(
    test_local_df,
    entries_by_doc_test_local,
    n=12,
    window=3,
    near_boundary_only=True,
    mixed_only=True,
    random_state=42,
)


Erreurs retenues: 136
DOC=410 | SENT=11 | doc_type=mixte | dist_frontiere=NA | TRUE=M | PRED=C | p(M)=0.0014
------------------------------------------------------------------------------------------------------------------------
    [sent=  8] TRUE=C | Je remercie également, naturellement, Monsieur le député des régions nordiques à la Douma et puis je voudrais souligner l'apport important et la passion, la générosité, l'intelligence mises par le <nom> dans la défense de cette grande cause.
    [sent=  9] TRUE=C | En effet, avec l'Académie Polaire, la Russie et Saint-Pétersbourg seront exemplaires des relations qui existent entre les différentes civilisations du monde et notamment le respect que l'on doit à ces peuples premiers dont on n'a pas encore estimé à sa juste valeur l'apport qu'ils peuvent faire à l'évolution du monde de demain.
    [sent= 10] TRUE=C | Donc, je vous remercie tous.
>>> [sent= 11] TRUE=M | Cela pourrait donner lieu à une méditation sur le terrorisme, sur cette v

## 12quater. Tests rapides sans réentraînement : varier la taille de la fenêtre au test

Ici, on garde **le même modèle déjà entraîné**, et on change seulement la taille de la **fenêtre de contexte** au moment de prédire sur `test_local`.

Cela permet de vérifier rapidement si :
- un **contexte plus petit** aide près des frontières,
- le contexte actuel est parfois **trop dominant**,
- votre hypothèse sur les documents mixtes semble correcte.

Ces tests sont **peu coûteux** car ils ne réentraînent pas le modèle.


In [22]:

def predict_probs_for_texts_with_trainer(trainer, tokenizer, texts, batch_size=128, max_length=512):
    hf_ds = HFDataset.from_dict({"text": texts})

    def _tok(ex):
        return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=max_length)

    tok_ds = hf_ds.map(_tok, batched=True, batch_size=batch_size)
    pred = trainer.predict(tok_ds)
    logits = pred.predictions
    probs = torch.nn.functional.softmax(torch.from_numpy(logits), dim=-1)[:, 1].numpy()
    return probs

def eval_fixed_window_sizes(sizes=(150, 200, 250, 350)):
    rows = []
    for size in sizes:
        texts = build_context_windows(test_local_entries, max_context_words=size)
        probs = predict_probs_for_texts_with_trainer(trainer, tokenizer, texts)
        preds = (probs > 0.5).astype(int)
        rows.append({
            "context_words": size,
            "f1_macro": f1_score(test_local_labels, preds, average="macro") * 100,
            "auc": roc_auc_score(test_local_labels, probs) * 100,
            "ap": average_precision_score(test_local_labels, probs) * 100,
            "recall_M": recall_score(test_local_labels, preds, pos_label=1) * 100,
            "precision_M": precision_score(test_local_labels, preds, pos_label=1, zero_division=0) * 100,
            "pred_M_rate": preds.mean() * 100,
        })
    return pd.DataFrame(rows).sort_values("f1_macro", ascending=False)

window_grid_results = eval_fixed_window_sizes(sizes=(120, 150, 200, 250, 350))
display(window_grid_results)


Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

,context_words,f1_macro,auc,ap,recall_M,precision_M,pred_M_rate
3,250,89.823036,97.673036,83.997179,90.609137,75.957447,15.997277
4,350,87.905754,97.207498,83.118067,85.025381,74.279379,15.350579
2,200,87.823954,97.523834,85.352503,89.974619,70.970971,17.001361
1,150,87.134529,97.395322,86.350817,89.847716,69.275930,17.392784
0,120,86.590127,97.295380,87.853902,88.324873,68.706811,17.239619


## 12quinquies. Test adaptatif local (diagnostic) sur les zones critiques

Ici, on fait un **test local diagnostique** :

- si une phrase est dans un **document mixte** ou **près d'une frontière**, on lui donne une **fenêtre plus petite** ;
- sinon on garde la fenêtre standard.

⚠️ Important :
- ce test utilise l'information des labels du `test_local` pour détecter les zones critiques ;
- il sert donc à **diagnostiquer l'idée**, pas à produire directement une soumission finale.

S'il aide nettement en local, alors cela justifie de concevoir ensuite une version plus réaliste.


In [23]:

def build_context_windows_adaptive_local(entries, default_words=350, critical_words=200, use_mixed_docs=True, use_near_boundary=True, boundary_k=1):
    # On s'appuie sur les colonnes déjà calculées dans test_local_df
    meta = test_local_df[["doc", "sent", "doc_type", "near_boundary_le_1", "near_boundary_le_3"]].copy()
    meta["critical"] = 0

    if use_mixed_docs:
        meta.loc[meta["doc_type"] == "mixte", "critical"] = 1

    if use_near_boundary:
        col = "near_boundary_le_1" if boundary_k == 1 else "near_boundary_le_3"
        meta.loc[meta[col] == 1, "critical"] = 1

    word_budget = {
        (int(r.doc), int(r.sent)): (critical_words if int(r.critical) == 1 else default_words)
        for r in meta.itertuples(index=False)
    }

    by_doc = defaultdict(list)
    for entry in entries:
        by_doc[entry["doc"]].append(entry)

    windows = []
    for doc_id in sorted(by_doc.keys()):
        sents = sorted(by_doc[doc_id], key=lambda x: x["sent"])
        for pos, entry in enumerate(sents):
            budget = word_budget[(int(entry["doc"]), int(entry["sent"]))]
            window = [entry["text"]]
            word_count = len(entry["text"].split())
            left, right = pos - 1, pos + 1

            while word_count < budget:
                added = False
                if left >= 0:
                    w = len(sents[left]["text"].split())
                    if word_count + w <= budget:
                        window.insert(0, sents[left]["text"])
                        word_count += w
                        left -= 1
                        added = True
                if right < len(sents):
                    w = len(sents[right]["text"].split())
                    if word_count + w <= budget:
                        window.append(sents[right]["text"])
                        word_count += w
                        right += 1
                        added = True
                if not added:
                    break
            windows.append(" ".join(window))
    return windows

def eval_adaptive_local(default_words=350, critical_words=200, use_mixed_docs=True, use_near_boundary=True, boundary_k=1):
    texts = build_context_windows_adaptive_local(
        test_local_entries,
        default_words=default_words,
        critical_words=critical_words,
        use_mixed_docs=use_mixed_docs,
        use_near_boundary=use_near_boundary,
        boundary_k=boundary_k,
    )
    probs = predict_probs_for_texts_with_trainer(trainer, tokenizer, texts)
    preds = (probs > 0.5).astype(int)
    return {
        "default_words": default_words,
        "critical_words": critical_words,
        "mixed_docs": use_mixed_docs,
        "near_boundary": use_near_boundary,
        "boundary_k": boundary_k,
        "f1_macro": f1_score(test_local_labels, preds, average="macro") * 100,
        "auc": roc_auc_score(test_local_labels, probs) * 100,
        "ap": average_precision_score(test_local_labels, probs) * 100,
        "recall_M": recall_score(test_local_labels, preds, pos_label=1) * 100,
        "precision_M": precision_score(test_local_labels, preds, pos_label=1, zero_division=0) * 100,
        "pred_M_rate": preds.mean() * 100,
    }

adaptive_rows = []
for crit in [120, 150, 200, 250]:
    adaptive_rows.append(eval_adaptive_local(default_words=350, critical_words=crit, use_mixed_docs=True, use_near_boundary=True, boundary_k=1))
    adaptive_rows.append(eval_adaptive_local(default_words=350, critical_words=crit, use_mixed_docs=False, use_near_boundary=True, boundary_k=1))
    adaptive_rows.append(eval_adaptive_local(default_words=350, critical_words=crit, use_mixed_docs=True, use_near_boundary=False, boundary_k=1))

adaptive_results = pd.DataFrame(adaptive_rows).sort_values("f1_macro", ascending=False)
display(adaptive_results)


Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

Map:   0%|          | 0/5876 [00:00<?, ? examples/s]

,default_words,critical_words,mixed_docs,near_boundary,boundary_k,f1_macro,auc,ap,recall_M,precision_M,pred_M_rate
0,350,120,True,True,1,49.824427,53.161016,14.462039,15.609137,13.225806,15.827093
2,350,120,True,False,1,49.824427,53.161016,14.462039,15.609137,13.225806,15.827093
8,350,200,True,False,1,49.540459,52.164444,14.733445,15.482234,12.815126,16.201498
6,350,200,True,True,1,49.540459,52.164444,14.733445,15.482234,12.815126,16.201498
9,350,250,True,True,1,49.385049,51.490755,14.343143,14.593909,12.513602,15.639891
11,350,250,True,False,1,49.385049,51.490755,14.343143,14.593909,12.513602,15.639891
5,350,150,True,False,1,49.212430,52.739301,14.684787,14.720812,12.288136,16.065351
3,350,150,True,True,1,49.212430,52.739301,14.684787,14.720812,12.288136,16.065351
4,350,150,False,True,1,48.948792,49.744472,13.340561,13.197970,11.751412,15.061266
7,350,200,False,True,1,48.918316,49.735792,13.248696,13.451777,11.738649,15.367597


## 12sexies. Réentraînement optionnel avec une autre taille de chunk

Cette partie permet de tester de vraies variantes de `v14` en réentraînant le modèle avec :

- `chunk_words = 200`
- `chunk_words = 250`
- `chunk_words = 350` (référence)

et éventuellement une autre taille de **fenêtre au test local**.

⚠️ C'est beaucoup plus coûteux que les cellules précédentes.  
Lance **une seule expérience à la fois**.


In [24]:

def run_retrain_experiment(
    chunk_words=250,
    context_words=250,
    model_name=MODEL_NAME,
    num_train_epochs=6,
    output_suffix=None,
    save_model=False,
):
    if output_suffix is None:
        output_suffix = f"cw{chunk_words}_ctx{context_words}"

    print(f"\n=== EXPERIMENT: chunk_words={chunk_words} | context_words={context_words} | model={model_name} ===")

    exp_train_chunks = build_chunks_from_entries(train_entries_split, max_words=chunk_words)
    exp_val_chunks   = build_chunks_from_entries(val_entries_split, max_words=chunk_words)

    exp_train_texts = [c["text"] for c in exp_train_chunks]
    exp_train_labels = np.array([c["label"] for c in exp_train_chunks])

    exp_val_texts = [c["text"] for c in exp_val_chunks]
    exp_val_labels = np.array([c["label"] for c in exp_val_chunks])

    exp_test_local_texts = build_context_windows(test_local_entries, max_context_words=context_words)

    print(f"Train chunks: {len(exp_train_chunks):,} | Val chunks: {len(exp_val_chunks):,}")
    print(f"Avg words/train chunk: {np.mean([len(c['text'].split()) for c in exp_train_chunks]):.1f}")

    exp_tokenizer = AutoTokenizer.from_pretrained(model_name)
    exp_train_dataset = ChunkDataset(exp_train_texts, exp_train_labels, exp_tokenizer, max_length=512)
    exp_val_dataset   = ChunkDataset(exp_val_texts, exp_val_labels, exp_tokenizer, max_length=512)

    exp_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    exp_model.to(device)

    exp_output_dir = os.path.join(OUTPUT_DIR, f"exp_{output_suffix}")
    os.makedirs(exp_output_dir, exist_ok=True)

    exp_args = TrainingArguments(
        output_dir=exp_output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-5,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="auc",
        greater_is_better=True,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )

    exp_trainer = Trainer(
        model=exp_model,
        args=exp_args,
        train_dataset=exp_train_dataset,
        eval_dataset=exp_val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    exp_trainer.train()
    exp_val_results = exp_trainer.evaluate()

    exp_probs_local = predict_probs_for_texts_with_trainer(exp_trainer, exp_tokenizer, exp_test_local_texts)
    exp_preds_local = (exp_probs_local > 0.5).astype(int)

    result = {
        "chunk_words": chunk_words,
        "context_words": context_words,
        "val_f1_macro": exp_val_results.get("eval_f1_macro"),
        "val_auc": exp_val_results.get("eval_auc"),
        "val_ap": exp_val_results.get("eval_ap"),
        "test_local_f1_macro": f1_score(test_local_labels, exp_preds_local, average="macro") * 100,
        "test_local_auc": roc_auc_score(test_local_labels, exp_probs_local) * 100,
        "test_local_ap": average_precision_score(test_local_labels, exp_probs_local) * 100,
        "test_local_recall_M": recall_score(test_local_labels, exp_preds_local, pos_label=1) * 100,
        "test_local_precision_M": precision_score(test_local_labels, exp_preds_local, pos_label=1, zero_division=0) * 100,
        "pred_M_rate": exp_preds_local.mean() * 100,
        "output_dir": exp_output_dir,
    }

    if save_model:
        exp_model_dir = os.path.join(MODEL_SAVE_DIR, f"exp_{output_suffix}")
        os.makedirs(exp_model_dir, exist_ok=True)
        exp_trainer.save_model(exp_model_dir)
        exp_tokenizer.save_pretrained(exp_model_dir)
        result["saved_model_dir"] = exp_model_dir

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

print("Fonction run_retrain_experiment(...) prête.")


Fonction run_retrain_experiment(...) prête.


In [ ]:

# Décommente UNE ligne à la fois.
exp_200 = run_retrain_experiment(chunk_words=200, context_words=200, num_train_epochs=6, output_suffix="chunk200_ctx200")
exp_250 = run_retrain_experiment(chunk_words=250, context_words=250, num_train_epochs=6, output_suffix="chunk250_ctx250")
exp_350 = run_retrain_experiment(chunk_words=350, context_words=350, num_train_epochs=6, output_suffix="chunk350_ctx350")

# Exemple pour comparer ensuite :
pd.DataFrame([exp_200, exp_250, exp_350])



=== EXPERIMENT: chunk_words=200 | context_words=200 | model=camembert/camembert-large ===
Train chunks: 5,822 | Val chunks: 692
Avg words/train chunk: 170.1


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: camembert/camembert-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Macro,Auc,Ap
1,1.566680,0.068233,96.691000,99.525000,98.379000
2,0.284109,0.224716,91.333000,99.331000,97.888000
3,0.166133,0.080717,97.181000,99.710000,99.045000
4,0.041247,0.099518,97.446000,99.637000,98.881000
5,0.010644,0.139640,96.375000,99.448000,98.577000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


=== EXPERIMENT: chunk_words=250 | context_words=250 | model=camembert/camembert-large ===
Train chunks: 4,713 | Val chunks: 562
Avg words/train chunk: 210.1


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: camembert/camembert-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Macro,Auc,Ap
1,1.465859,0.070268,96.719000,99.470000,98.315000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 13. Optionnel — Réentraînement final sur tout le corpus learn
À lancer seulement quand vous êtes sûrs de la recette.

In [25]:

RETRAIN_ON_FULL_LEARN = False

if RETRAIN_ON_FULL_LEARN:
    full_chunks = build_chunks_from_entries(train_entries, max_words=MAX_WORDS)
    full_texts = [c["text"] for c in full_chunks]
    full_labels = np.array([c["label"] for c in full_chunks])

    full_dataset = ChunkDataset(full_texts, full_labels, tokenizer, max_length=512)

    final_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    final_model.to(device)

    final_args = TrainingArguments(
        output_dir=OUTPUT_DIR + "_full",
        eval_strategy="no",
        save_strategy="epoch",
        learning_rate=1e-5,
        num_train_epochs=10,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_steps=100,
        save_total_limit=2,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )

    final_trainer = Trainer(
        model=final_model,
        args=final_args,
        train_dataset=full_dataset,
    )

    final_trainer.train()
    final_trainer.save_model(MODEL_SAVE_DIR + "_full")
    tokenizer.save_pretrained(MODEL_SAVE_DIR + "_full")
    print("Modèle final sauvegardé dans:", MODEL_SAVE_DIR + "_full")


## 14. Prédiction finale sur le vrai test du prof
Cette cellule n'est à lancer que si le fichier test est présent.

In [26]:

RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    assert Path(TEST_FILE).exists(), f"Fichier introuvable: {TEST_FILE}"

    # Choisir quel modèle charger
    MODEL_FOR_INFERENCE = MODEL_SAVE_DIR
    if Path(MODEL_SAVE_DIR + "_full").exists():
        MODEL_FOR_INFERENCE = MODEL_SAVE_DIR + "_full"

    infer_tokenizer = AutoTokenizer.from_pretrained(MODEL_FOR_INFERENCE)
    infer_model = AutoModelForSequenceClassification.from_pretrained(MODEL_FOR_INFERENCE).to(device)

    test_entries = read_test_corpus(TEST_FILE)
    print("Test phrases:", len(test_entries))

    test_window_texts = build_context_windows(test_entries, max_context_words=MAX_CONTEXT_WORDS)
    test_hf = HFDataset.from_dict({"text": test_window_texts})

    def tok_final(ex):
        return infer_tokenizer(ex["text"], padding="max_length", truncation=True, max_length=512)

    test_tok = test_hf.map(tok_final, batched=True, batch_size=128)

    infer_trainer = Trainer(model=infer_model)
    pred = infer_trainer.predict(test_tok)
    logits = pred.predictions
    p_mitterrand = torch.nn.functional.softmax(torch.from_numpy(logits), dim=-1)[:, 1].numpy()

    submission_path = Path(SUBMISSION_DIR) / "submission-pres-v14-corrige.csv"
    with open(submission_path, "w", encoding="utf-8") as f:
        for p in p_mitterrand:
            f.write(f"{float(p)}\n")

    print("Soumission écrite dans:", submission_path)
    print(f"Lignes: {len(p_mitterrand)}")
    print(f"Mitterrand prédits (>0.5): {int(np.sum(p_mitterrand > 0.5))}")


## 15. Nettoyage

In [ ]:

gc.collect()
torch.cuda.empty_cache()
print("Mémoire nettoyée.")
